# Modul 06 · Notebook 07 — Grounding & Topik

Dua rail terakhir, keduanya soal **kepercayaan jawaban**:
- 🔍 **Grounding / anti-halusinasi** (pilar *Transparency*) — pastikan jawaban **didukung** konteks yang diambil (RAG nb03), bukan mengarang.
- 🎯 **Topical** — jaga bot tetap pada misinya (AI & ekosistem NVIDIA), tak melenceng.

> 🔑 Pakai `NVIDIA_API_KEY` (Colab Secrets).

In [1]:
!pip -q install openai
import os
if not os.path.exists("nvidia_utils.py"):
    !wget -q https://raw.githubusercontent.com/chmdznr/navasena-gen-ml-course/module06-rework/06_nvidia_ecosystem/tools/nvidia_utils.py
from google.colab import userdata
os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
print("Key termuat:", bool(os.environ.get("NVIDIA_API_KEY")))

Key termuat: True


## Bagian A — Grounding (self-check-facts)

Di nb03, RAG menjawab **berdasarkan konteks**. Tapi model bisa tetap **mengarang** di luar konteks (halusinasi). *Output rail* **grounding** memeriksa: apakah klaim jawaban **benar-benar didukung** konteks? Ini mewujudkan pilar **Transparency** — jawaban bisa ditelusuri ke sumbernya.

In [2]:
from nvidia_utils import nim_client, nim_chat
client = nim_client()
MODEL = "nvidia/nemotron-3-nano-30b-a3b"

KONTEKS = ("Quantization 4-bit memangkas memori model sehingga model besar muat di GPU kecil. "
           "Tensor Core mempercepat komputasi FP16 di GPU NVIDIA.")

def grounded(klaim, konteks):
    out = nim_chat(client, MODEL, max_tokens=8, messages=[
        {"role": "system", "content": "Nilai apakah KLAIM didukung PENUH oleh KONTEKS. "
                                       "Jika info tidak ada di konteks -> No."},
        {"role": "user", "content": f"KONTEKS: {konteks}\nKLAIM: {klaim}\n"
                                     "Apakah klaim didukung konteks? Jawab HANYA Yes atau No."}])
    return out.strip().lower().startswith("yes")

for klaim in ["Quantization 4-bit menghemat memori model.",
              "Tensor Core mempercepat FP16 di GPU NVIDIA.",
              "Quantization 4-bit mempercepat koneksi internet."]:
    print(f"{'GROUNDED  ' if grounded(klaim, KONTEKS) else 'HALUSINASI'} <- {klaim}")

GROUNDED   <- Quantization 4-bit menghemat memori model.
GROUNDED   <- Tensor Core mempercepat FP16 di GPU NVIDIA.
HALUSINASI <- Quantization 4-bit mempercepat koneksi internet.


> ✅ Klaim yang **tidak ada** di konteks ditandai **HALUSINASI**. Gabungkan dengan sitasi `[n]` (nb03) → jawaban yang **bisa dipertanggungjawabkan**.

## Bagian B — Topical rail

Bot tanya-jawab AI sebaiknya **menolak** keluar topik (saham, cuaca, gosip) — supaya tak disalahgunakan & tak memberi nasihat di luar kompetensinya. Kita klasifikasi topik lalu jawab/menolak.

In [3]:
def is_on_topic(p):
    out = nim_chat(client, MODEL, max_tokens=8, messages=[
        {"role": "system", "content": "Klasifikasikan topik. Jawab 'AI' jika pertanyaan tentang kecerdasan buatan, "
                                       "machine learning, LLM, GPU, atau NVIDIA. Jawab 'LAIN' untuk topik lain "
                                       "(saham, cuaca, gosip, resep, politik, dll)."},
        {"role": "user", "content": f'Pertanyaan: "{p}"\nTopik? Jawab HANYA satu kata: AI atau LAIN.'}])
    return out.strip().lower().startswith("ai")

def jawab(p):
    if not is_on_topic(p):
        return "[DITOLAK] Maaf, saya hanya membantu seputar AI & ekosistem NVIDIA."
    return nim_chat(client, MODEL, max_tokens=80, messages=[{"role": "user", "content": p}])

for q in ["Apa itu guardrail untuk LLM?", "Rekomendasikan saham yang bagus dong.", "Bagaimana cuaca besok?"]:
    print(f"{'ON ' if is_on_topic(q) else 'OFF'} | {q}\n      -> {jawab(q)[:100]}\n")

ON  | Apa itu guardrail untuk LLM?
      -> **Guardrail** (atau *safety guardrail*) dalam konteks **Large Language Model (LLM)** adalah mekanism

OFF | Rekomendasikan saham yang bagus dong.
      -> [DITOLAK] Maaf, saya hanya membantu seputar AI & ekosistem NVIDIA.

OFF | Bagaimana cuaca besok?
      -> [DITOLAK] Maaf, saya hanya membantu seputar AI & ekosistem NVIDIA.



## Bentuk deklaratif di NeMo Guardrails

Logika di atas diformalkan NeMo sebagai **output rail** (grounding) dan **dialog rail** (topical, ditulis dalam *Colang*):

```yaml
rails:
  output:
    flows: [ self check facts ]      # grounding: jawaban harus didukung konteks
```
```
# Colang (dialog rail topical)
define user ask off topic
  "rekomendasikan saham"
  "bagaimana cuaca hari ini"
define bot refuse off topic
  "Maaf, saya hanya membantu seputar AI & ekosistem NVIDIA."
define flow
  user ask off topic
  bot refuse off topic
```

> ⚠️ **Catatan model reasoning:** rail `self check facts` & dialog rail menjalankan LLM internal. Dengan model *reasoning* (Nemotron), jejak berpikir bisa membuat hasil lambat/keliru — sama seperti nb04. Solusi: matikan reasoning (`enable_thinking: false`) **dan/atau** demokan logikanya lewat panggilan langsung (seperti di atas), seperti yang kita lakukan di kelas ini.

## Yang kita pelajari & langkah berikut

- **Grounding** memverifikasi jawaban didukung konteks → anti-halusinasi + Transparency.
- **Topical rail** menjaga bot pada misinya.
- Keduanya melengkapi rail nb04–06; semua diformalkan NeMo secara deklaratif.

**Berikutnya:** **nb08** Capstone Deploy — bungkus semua rail (PII + jailbreak + grounding + topik) ke **FastAPI `/ask`** yang siap pakai.

> 🔗 Grounding menyambung ke sitasi RAG Modul 05 — *jawaban yang bisa ditelusuri = AI yang bisa dipercaya*.

## 🧪 Latihan

1. Tambah klaim baru yang **sebagian benar** ke `grounded` — apakah ditandai grounded atau halusinasi?
2. Pertajam `is_on_topic` agar **menerima** pertanyaan tentang *data science* juga.
3. Gabungkan: tolak dulu yang off-topic, baru cek grounding untuk yang on-topic.